### Setting path & importing libraries

In [ ]:
import sys
from pathlib import Path

# Add the src directory to the Python path to allow imports from there
# if you installed the bioairmet package then you can import directly without modifying sys.path, but this allows running the script without installing the package
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# importing the functions to clean the unlabeled data
from bioairmet.data.clean_unlabeled_data import * #, get_category_mapping
from bioairmet.data.MCH_helpers import *

In [34]:
import os
import json
import torch
import pandas as pd
import h5py
import numpy as np
import matplotlib.pyplot as plt
# inline plotting in Jupyter notebook
%matplotlib inline

## Reading data

### Labelled data 
Erb, Sophie, et al. "Pollen holographic images and light-induced fluorescence measurements at the species level." Scientific Data 12.1 (2025): 786

Dataset:
https://zenodo.org/records/15186184

In [ ]:
# path to the raw files
root_path = "/media/data/METAS_data_Poleno/data_untar/MCH_datasets_pollen_2023-2024/"
data_clean_ids = os.path.join(root_path, "data_clean_ids")
data_zip_files = os.path.join(root_path, "data_zip_files")
meta_data_path = os.path.join(root_path, 'MCH_metadata_pollen_2023-2024.xlsx')

In [ ]:
print("Starting data processing...")
print(f"1. Getting directories and files from: {root_path}")
data = get_directories_and_files(root_path)
# data2 = get_directories_and_files(path2)

print("2. Extracting event names and image paths...")
data_fast = get_event_name_fast(data, root_path)
print("3. Converting to DataFrame and filtering invalid entries...")
data_final, df_all_events = get_df(data_fast)
print("4. Adding labels to the final DataFrame after Cleaning...")
final_df = get_final_df_with_labels(data_clean_ids, data_final, meta_data_path)

In [42]:
# Creating a mapping from category names to category numbers based on the categories present in the final_df (after cleaning) and the metadata, excluding "Garbage" class which will be assigned the last category number
class_count = final_df[final_df['class']!="Garbage"]['class'].value_counts().reset_index()
all_categories = {"Garbage": len(class_count['class'])} # assign "Garbage" class the last category number, which is the length of the existing categories

for i, cls in enumerate(class_count['class']):
    all_categories[cls] = i
# save the category mapping to a text file for later use in training and validation
category_mapping_save_path = "./category_num_map.txt"
with open(category_mapping_save_path, "w") as f:
    f.write("# category_num mapping, seperated by a tab\n")
    for cat_name, cat_num in all_categories.items():
            f.write(f"{cat_name}\t{cat_num}\n")

In [32]:
# a function to read relative fl spectra from the event_path
def readevent_spectra(event_path):
    """
    Read and process fluorescence spectra data from a JSON file.

    Args:
        event_path (str): Path to the JSON file

    Returns:
        torch.Tensor: Tensor of shape (3, 5) containing the spectra data,
                        or zeros if data cannot be read
    """
    # Default return value if anything fails
    default_tensor = torch.zeros((3, 5), dtype=torch.float32)

    try:
        # Safely load JSON data
        with open(event_path, 'r') as file:
            data = json.load(file)

        # Early return if data is None or empty
        if not data:
            return default_tensor

        # Try new format first
        if isinstance(data, dict) and 'computed_data' in data:
            try:
                spectra_path = data['computed_data']['fluorescence']['processed_data']['spectra']
                if 'relative_spectra' in spectra_path:
                    return torch.tensor(spectra_path['relative_spectra'], dtype=torch.float32).view(3, 5)
            except (KeyError, TypeError):
                pass

        # Try old format if new format fails
        if isinstance(data, dict) and 'computedData' in data:
            try:
                if 'fluorescenceSpectra' in data['computedData']:
                    spectra = data['computedData']['fluorescenceSpectra']
                    if 'relative_spectra' in spectra:
                        return torch.tensor(spectra['relative_spectra'], dtype=torch.float32).view(3, 5)
                    elif 'mean_average_scaled' in spectra:
                        mean_data = spectra['mean_average_scaled']
                        values = [
                            mean_data.get('mean_280', [0] * 5),
                            mean_data.get('mean_365', [0] * 5),
                            mean_data.get('mean_405', [0] * 5)
                        ]
                        return torch.tensor(values, dtype=torch.float32)
            except (KeyError, TypeError):
                pass

    except (json.JSONDecodeError, FileNotFoundError, Exception) as e:
        print(f"Error reading spectra from {event_path}: {str(e)}")

    return default_tensor

In [38]:
final_df['relative_spectra'] = final_df['event'].apply(readevent_spectra)

In [45]:
final_df['category_number'] = final_df['class'].map(all_categories)

In [49]:
final_df.rename(columns={'class': 'category', 'category_number': 'category_num'}, inplace=True)

In [ ]:
final_df.head()

### Splitting to train and test

In [59]:
from sklearn.model_selection import train_test_split

# Function to split the DataFrame
def split_dataframe(df):
    train_list = []
    test_list = []
    
    # Group by category
    for category_name, group in df.groupby('category'):
        
        # Split each group into train and test sets
        train, test = train_test_split(group, test_size=0.25, random_state=42)
        
        # Append to the respective lists
        train_list.append(train)
        test_list.append(test)
    
    # Concatenate the lists back into DataFrames
    train_df = pd.concat(train_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)
    
    return train_df, test_df

In [60]:
train_df, test_df = split_dataframe(final_df)

In [54]:
final_df.shape

(1037928, 5)

In [62]:
final_df['category'].value_counts()

category
Corylus         208927
Garbage         164697
Betula           77512
Alnus            71977
Taxus            57530
Quercus          34396
Trisetum         30599
Plantago         26983
Carpinus         26353
Urtica           23926
Juglans          23677
Cedrus           19965
Fraxinus         17140
Ambrosia         16994
Sambucus         16880
Cryptomeria      16266
Rumex            15811
Artemisia        15474
Trachycarpus     15089
Carex            15085
Platanus         14386
Pinus            14373
Picea            14356
Pseudotsuga      13845
Bromus           13344
Dactylis         12312
Liquidambar      12246
Larix            10859
Alopecurus        9839
Abies             9115
Lolium            9103
Castanea          4214
Fagus             4012
Tilia              282
Ostrya             190
Acer               171
Name: count, dtype: int64

In [61]:
test_df['category'].value_counts()

category
Corylus         52232
Garbage         41175
Betula          19378
Alnus           17995
Taxus           14383
Quercus          8599
Trisetum         7650
Plantago         6746
Carpinus         6589
Urtica           5982
Juglans          5920
Cedrus           4992
Fraxinus         4285
Ambrosia         4249
Sambucus         4220
Cryptomeria      4067
Rumex            3953
Artemisia        3869
Trachycarpus     3773
Carex            3772
Platanus         3597
Pinus            3594
Picea            3589
Pseudotsuga      3462
Bromus           3336
Dactylis         3078
Liquidambar      3062
Larix            2715
Alopecurus       2460
Abies            2279
Lolium           2276
Castanea         1054
Fagus            1003
Tilia              71
Ostrya             48
Acer               43
Name: count, dtype: int64

In [ ]:
save_df_to_hdf5(train_df, "train_data.h5")
save_df_to_hdf5(test_df, "test_data.h5")